<a href="https://colab.research.google.com/github/FoldAndFunction/rfdiffusion-ISN2026/blob/main/rfdiffusion_course_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**RFdiffusion v1.1.1** **- ISN, Mendoza, 2026**
RFdiffusion is a method for structure generation, with or without conditional information (a motif, target etc). It can perform a whole range of protein design challenges as we have outlined in the RFdiffusion [manuscript](https://www.biorxiv.org/content/10.1101/2022.12.09.519842v2).


For **instructions**, see end of Notebook.

**<font color="red">NOTE:</font>**  This is tagged v1.1.1 of the notebook, this notebook may break in the future when colab updates. For latest version see [main](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb) branch.

Press Cmd + Shift + P → Change runtime version → 2026.07

Additional Notebooks:

- See [diffusion_foldcond](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/v1.1.1/rf/examples/diffusion_foldcond.ipynb) for fold conditioning functionality.

- See [original version](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/v1.1.1/rf/examples/diffusion_ori.ipynb) of this notebook (from 31Mar2023).


In [2]:
#@title setup **RFdiffusion** (reliable Colab setup; ~10-20 min first run)
%%time
import os, sys, time, signal, random, string, re
import collections, collections.abc
import gzip, hashlib, importlib.metadata, pathlib, shutil, subprocess, tarfile

print(sys.version)

# Reproducible repository revisions checked on 2026-08-26.
RFDIFFUSION_REV = "597d37f2a686e23941440fddf6daa4cb778e7bc7"
COLABDESIGN_REV = "e31a56fe1d9b4de25c8697f3a28b75892941cc72"
TORCH_VERSION = "2.4.0"
DGL_VERSION = "2.4.0+cu124"
DOWNLOAD_AF2_PARAMS = True
RUN_SMOKE_TEST = True

os.environ["DGLBACKEND"] = "pytorch"

# Compatibility for older dependencies on Python 3.12.
for _name in ("Mapping", "Iterable", "MutableMapping", "Sequence"):
  if not hasattr(collections, _name):
    setattr(collections, _name, getattr(collections.abc, _name))

def run_checked(args, cwd=None, quiet=False):
  """Run a command and stop immediately with its real output on failure."""
  shown = " ".join(map(str, args))
  if not quiet:
    print(f"\n$ {shown}")

  result = subprocess.run(
      list(map(str, args)),
      cwd=cwd,
      text=True,
      stdout=subprocess.PIPE if quiet else None,
      stderr=subprocess.STDOUT if quiet else None)

  if result.returncode != 0:
    detail = (result.stdout or "").strip()
    raise RuntimeError(
        f"Command failed ({result.returncode}): {shown}\n{detail}")

  return result.stdout if quiet else ""

def installed_version(package):
  try:
    return importlib.metadata.version(package)
  except importlib.metadata.PackageNotFoundError:
    return None

def download(url, destination, min_bytes=1024):
  """Download a file and reject empty or truncated results."""
  destination = pathlib.Path(destination)
  destination.parent.mkdir(parents=True, exist_ok=True)

  if destination.is_file() and destination.stat().st_size >= min_bytes:
    print(
        f"Using existing {destination} "
        f"({destination.stat().st_size:,} bytes)")
    return

  destination.unlink(missing_ok=True)
  pathlib.Path(str(destination) + ".aria2").unlink(missing_ok=True)

  run_checked([
      "aria2c",
      "--console-log-level=warn",
      "--summary-interval=0",
      "--allow-overwrite=true",
      "--auto-file-renaming=false",
      "--check-integrity=true",
      "-x", "16",
      "-s", "16",
      "-d", str(destination.parent),
      "-o", destination.name,
      url
  ])

  if not destination.is_file() or destination.stat().st_size < min_bytes:
    raise RuntimeError(
        f"Download is missing or truncated: {destination}")

def sha256_file(path, chunk_size=8 * 1024 * 1024):
  digest = hashlib.sha256()

  with open(path, "rb") as handle:
    for chunk in iter(lambda: handle.read(chunk_size), b""):
      digest.update(chunk)

  return digest.hexdigest()

# Runtime validation.
if sys.version_info[:2] != (3, 12):
  raise RuntimeError(
      f"This setup requires Colab Python 3.12; "
      f"found {sys.version.split()[0]}. "
      "Select Runtime > Change runtime type > Runtime version 2026.07.")

if shutil.which("nvidia-smi") is None:
  raise RuntimeError(
      "No NVIDIA GPU detected. "
      "Select Runtime > Change runtime type > GPU.")

# Install aria2 if necessary.
if shutil.which("aria2c") is None:
  run_checked(["apt-get", "update", "-qq"])
  run_checked(["apt-get", "install", "-y", "-qq", "aria2"])

# DGL 2.4.0 CUDA wheel requires PyTorch 2.4.0.
if installed_version("torch") != TORCH_VERSION:
  run_checked([
      sys.executable,
      "-m", "pip", "install",
      "-q",
      "--upgrade",
      "--force-reinstall",
      f"torch=={TORCH_VERSION}",
      "torchvision==0.19.0",
      "torchaudio==2.4.0",
      "--index-url",
      "https://download.pytorch.org/whl/cu124"
  ])

if installed_version("dgl") != DGL_VERSION:
  run_checked([
      sys.executable,
      "-m", "pip", "install",
      "-q",
      "--upgrade",
      "--force-reinstall",
      "--no-dependencies",
      f"dgl=={DGL_VERSION}",
      "-f",
      "https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html"
  ])

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy==2.0.2",
    "scipy==1.16.3",
    "jedi",
    "omegaconf",
    "hydra-core",
    "icecream",
    "pyrsistent",
    "pynvml",
    "decorator",
    "e3nn==0.5.5",
    "opt_einsum_fx",
    "py3Dmol",
    "absl-py",
    "biopython",
    "chex",
    "dm-haiku",
    "dm-tree",
    "immutabledict",
    "ml-collections",
    "optax",
    "joblib",
])

run_checked([
    sys.executable,
    "-m", "pip", "install",
    "-q",
    "--upgrade",
    "git+https://github.com/NVIDIA/dllogger#egg=dllogger"
])

# Clone and pin RFdiffusion.
rf_root = pathlib.Path("RFdiffusion")

if rf_root.exists() and not (rf_root / ".git").is_dir():
  raise RuntimeError(
      "RFdiffusion exists but is incomplete. "
      "Factory-reset the Colab runtime and rerun.")

if not rf_root.exists():
  run_checked([
      "git",
      "clone",
      "https://github.com/sokrypton/RFdiffusion.git",
      str(rf_root)
  ])

run_checked([
    "git",
    "fetch",
    "--depth", "1",
    "origin",
    RFDIFFUSION_REV
], cwd=rf_root)

run_checked([
    "git",
    "checkout",
    "--detach",
    RFDIFFUSION_REV
], cwd=rf_root)

# Install pinned ColabDesign.
run_checked([
    sys.executable,
    "-m", "pip", "install",
    "-q",
    "--upgrade",
    "--force-reinstall",
    "--no-dependencies",
    f"git+https://github.com/sokrypton/ColabDesign.git@{COLABDESIGN_REV}"
])

# Install the SE(3) Transformer.
run_checked([
    sys.executable,
    "-m", "pip", "install",
    "-q",
    "--upgrade",
    "--force-reinstall",
    "--no-dependencies",
    "."
], cwd=rf_root / "env" / "SE3Transformer")

# AnAnaS is optional and only used for automatic symmetry detection.
ANANAS_AVAILABLE = False

try:
  download(
      "https://files.ipd.uw.edu/krypton/ananas",
      "ananas",
      10_000)

  pathlib.Path("ananas").chmod(0o755)
  ANANAS_AVAILABLE = True

except RuntimeError:
  print(
      "WARNING: AnAnaS is unavailable; "
      "automatic symmetry detection is disabled.")

# RFdiffusion model checkpoints.
models = {
    "Base_ckpt.pt":
      "https://files.ipd.uw.edu/pub/RFdiffusion/"
      "6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt",

    "Complex_base_ckpt.pt":
      "https://files.ipd.uw.edu/pub/RFdiffusion/"
      "e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt",

    "Complex_beta_ckpt.pt":
      "https://files.ipd.uw.edu/pub/RFdiffusion/"
      "f572d396fae9206628714fb2ce00f72e/Complex_beta_ckpt.pt",
}

model_dir = rf_root / "models"

for filename, url in models.items():
  download(
      url,
      model_dir / filename,
      1_000_000)

# Precomputed IGSO(3) schedule for diffuser.T=50.
# The source revision and checksum are fixed.
schedule_name = (
    "T_50_omega_1000_min_sigma_0_02_"
    "min_b_1_5_max_b_2_5_schedule_linear.pkl"
)

schedule_sha256 = (
    "c96eb0d4be40ffe2489c3e41f6f4fc0f"
    "e92a0dea7f98410a1486dcf94b764217"
)

schedule_path = rf_root / "schedules" / schedule_name

schedule_url = (
    "https://huggingface.co/GlandVergil/RFdiffusion/resolve/"
    "3cdaa7d9e22dbdf085abbd16f17b4dc31995ce4d/"
    "schedules/"
    + schedule_name
)

download(
    schedule_url,
    schedule_path,
    8_000_000)

actual_schedule_sha256 = sha256_file(schedule_path)

if actual_schedule_sha256 != schedule_sha256:
  schedule_path.unlink(missing_ok=True)

  raise RuntimeError(
      "RFdiffusion T=50 schedule checksum mismatch: "
      f"expected {schedule_sha256}, "
      f"got {actual_schedule_sha256}")

print(
    f"{schedule_name}: "
    f"{schedule_path.stat().st_size:,} bytes "
    f"sha256={actual_schedule_sha256}")

# AlphaFold parameters, required by the later validation cell.
params_dir = pathlib.Path("params")
params_marker = params_dir / "done.txt"

if DOWNLOAD_AF2_PARAMS and not params_marker.is_file():
  params_dir.mkdir(exist_ok=True)

  af_archive = pathlib.Path(
      "alphafold_params_2022-12-06.tar")

  download(
      "https://storage.googleapis.com/alphafold/"
      "alphafold_params_2022-12-06.tar",
      af_archive,
      1_000_000_000)

  print("Extracting AlphaFold parameters...")

  with tarfile.open(af_archive) as archive:
    archive.extractall(params_dir)

  if not any(params_dir.glob("*.npz")):
    raise RuntimeError(
        "AlphaFold parameter extraction produced no .npz files.")

  params_marker.write_text(
      "complete\n",
      encoding="utf-8")

  af_archive.unlink()

# Make RFdiffusion importable.
if str(rf_root.resolve()) not in sys.path:
  sys.path.insert(0, str(rf_root.resolve()))

# Import and GPU validation.
import torch
import dgl

from dgl.nn.pytorch import AvgPooling, MaxPooling
from inference.utils import parse_pdb

if torch.__version__.split("+")[0] != TORCH_VERSION:
  raise RuntimeError(
      f"Expected torch {TORCH_VERSION}, "
      f"but imported {torch.__version__}. "
      "Restart the runtime and rerun.")

if not torch.cuda.is_available():
  raise RuntimeError(
      "PyTorch cannot see the GPU after installation.")

print("\nValidated environment")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("DGL:", dgl.__version__)
print("GPU:", torch.cuda.get_device_name(0))

for model_path in sorted(model_dir.glob("*.pt")):
  digest = sha256_file(model_path)

  print(
      f"{model_path.name}: "
      f"{model_path.stat().st_size:,} bytes "
      f"sha256={digest}")

# Real RFdiffusion GPU smoke test.
if RUN_SMOKE_TEST:
  smoke_root = pathlib.Path("_rf_smoke")

  shutil.rmtree(
      smoke_root,
      ignore_errors=True)

  smoke_root.mkdir()
  smoke_log = smoke_root / "smoke.log"

  command = [
      sys.executable,
      str(rf_root / "run_inference.py"),
      f"inference.output_prefix={smoke_root / 'design'}",
      "inference.num_designs=1",
      "diffuser.T=50",
      "contigmap.contigs=[20-20]",
      "inference.dump_pdb=False"
  ]

  with smoke_log.open(
      "w",
      encoding="utf-8") as handle:

    result = subprocess.run(
        command,
        text=True,
        stdout=handle,
        stderr=subprocess.STDOUT)

  expected_output = smoke_root / "design_0.pdb"

  if result.returncode != 0 or not expected_output.is_file():
    detail = smoke_log.read_text(
        encoding="utf-8",
        errors="replace")[-8000:]

    raise RuntimeError(
        f"RFdiffusion smoke test failed:\n{detail}")

  print("RFdiffusion 50-step GPU smoke test: PASS")

  shutil.rmtree(smoke_root)

# Notebook utilities.
from google.colab import files

import json
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import py3Dmol

from IPython.display import display, HTML
from inference.utils import parse_pdb

from colabdesign.rf.utils import get_ca
from colabdesign.rf.utils import (
    fix_contigs,
    fix_partial_contigs,
    fix_pdb,
    sym_it,
)

from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

def get_pdb(pdb_code=None):
  if pdb_code is None or pdb_code == "":
    upload_dict = files.upload()
    pdb_string = upload_dict[
        list(upload_dict.keys())[0]]

    with open("tmp.pdb", "wb") as out:
      out.write(pdb_string)

    return "tmp.pdb"

  elif os.path.isfile(pdb_code):
    return pdb_code

  elif len(pdb_code) == 4:
    pdb_code = pdb_code.upper()
    pdb_path = pathlib.Path(f"{pdb_code}.pdb1")

    if not pdb_path.is_file():
      gz_path = pathlib.Path(
          f"{pdb_code}.pdb1.gz")

      download(
          f"https://files.rcsb.org/download/"
          f"{pdb_code}.pdb1.gz",
          gz_path,
          100)

      with gzip.open(gz_path, "rb") as source:
        with pdb_path.open("wb") as target:
          shutil.copyfileobj(source, target)

      gz_path.unlink()

    return str(pdb_path)

  else:
    for version in (4, 3):
      af_path = pathlib.Path(
          f"AF-{pdb_code}-F1-model_v{version}.pdb")

      try:
        download(
            "https://alphafold.ebi.ac.uk/files/"
            f"{af_path.name}",
            af_path,
            100)

        return str(af_path)

      except RuntimeError:
        af_path.unlink(missing_ok=True)

    raise RuntimeError(
        f"Could not download AlphaFold DB entry: {pdb_code}")

def run_ananas(pdb_str, path, sym=None):
  if not ANANAS_AVAILABLE:
    raise RuntimeError(
        "Automatic symmetry detection is unavailable. "
        "Use symmetry=none, cyclic, or dihedral.")

  pdb_filename = (
      f"outputs/{path}/ananas_input.pdb")

  out_filename = (
      f"outputs/{path}/ananas.json")

  with open(pdb_filename, "w") as handle:
    handle.write(pdb_str)

  cmd = [
      "./ananas",
      pdb_filename,
      "-u",
      "-j",
      out_filename,
  ]

  if sym is not None:
    cmd.append(str(sym))

  run_checked(cmd)

  try:
    with open(out_filename, "r") as handle:
      out = json.loads(handle.read())

    results = out[0]
    AU = out[-1]["AU"]
    group = AU["group"]
    chains = AU["chain names"]
    rmsd = results["Average_RMSD"]

    print(
        f"AnAnaS detected {group} symmetry "
        f"at RMSD:{rmsd:.3}")

    C = np.array(
        results["transforms"][0]["CENTER"])

    A = [
        np.array(transform["AXIS"])
        for transform in results["transforms"]
    ]

    new_lines = []

    for line in pdb_str.split("\n"):
      if line.startswith("ATOM"):
        chain = line[21:22]

        if chain in chains:
          x = np.array([
              float(line[i:i + 8])
              for i in [30, 38, 46]
          ])

          if group[0] == "c":
            x = sym_it(x, C, A[0])

          if group[0] == "d":
            x = sym_it(x, C, A[1], A[0])

          coord_str = "".join(
              f"{coordinate:8.3f}"
              for coordinate in x)

          new_lines.append(
              line[:30]
              + coord_str
              + line[54:])

      else:
        new_lines.append(line)

    return results, "\n".join(new_lines)

  except (
      OSError,
      ValueError,
      KeyError,
      IndexError,
      TypeError,
      json.JSONDecodeError,
  ):
    return None, pdb_str

def run(command, steps, num_designs=1, visual="none"):

  def run_command_and_get_pid(command):
    pid_file = "/dev/shm/pid"

    log_file = pathlib.Path(
        "rfdiffusion_last_run.log").resolve()

    shell_command = (
        f'nohup {command} > "{log_file}" 2>&1 '
        f'& echo $! > {pid_file}'
    )

    status = os.system(shell_command)

    if status != 0:
      raise RuntimeError(
          f"Could not launch RFdiffusion; "
          f"see {log_file}")

    with open(pid_file, "r") as handle:
      pid = int(handle.read().strip())

    os.remove(pid_file)

    return pid, log_file

  def is_process_running(pid):
    try:
      os.kill(pid, 0)
    except OSError:
      return False
    else:
      return True

  run_output = widgets.Output()

  progress = widgets.FloatProgress(
      min=0,
      max=1,
      description="running",
      bar_style="info")

  display(
      widgets.VBox([
          progress,
          run_output,
      ]))

  # Clear previous temporary frames.
  for step in range(steps):
    frame_path = f"/dev/shm/{step}.pdb"

    if os.path.isfile(frame_path):
      os.remove(frame_path)

  pid, log_file = run_command_and_get_pid(command)

  try:
    fail = False

    for _ in range(num_designs):
      for step in range(steps):
        wait = True

        while wait and not fail:
          time.sleep(0.1)

          frame_path = f"/dev/shm/{step}.pdb"

          if os.path.isfile(frame_path):
            with open(frame_path) as handle:
              pdb_str = handle.read()

            if pdb_str.rstrip().endswith("TER"):
              wait = False

            elif not is_process_running(pid):
              fail = True

          elif not is_process_running(pid):
            fail = True

        if fail:
          progress.bar_style = "danger"
          progress.description = "failed"

          if log_file.is_file():
            print(
                "\nRFdiffusion error log "
                "(last 80 lines):")

            lines = log_file.read_text(
                encoding="utf-8",
                errors="replace").splitlines()

            print("\n".join(lines[-80:]))

          break

        progress.value = (
            (step + 1) / steps)

        if visual != "none":
          with run_output:
            run_output.clear_output(wait=True)

            if visual == "image":
              xyz, bfact = get_ca(
                  frame_path,
                  get_bfact=True)

              fig = plt.figure()
              fig.set_dpi(100)
              fig.set_figwidth(6)
              fig.set_figheight(6)

              axis = fig.add_subplot(111)
              axis.set_xticks([])
              axis.set_yticks([])

              plot_pseudo_3D(
                  xyz,
                  c=bfact,
                  cmin=0.5,
                  cmax=0.9,
                  ax=axis)

              plt.show()

            elif visual == "interactive":
              view = py3Dmol.view(
                  js="https://3dmol.org/build/3Dmol.js")

              view.addModel(
                  pdb_str,
                  "pdb")

              view.setStyle({
                  "cartoon": {
                      "colorscheme": {
                          "prop": "b",
                          "gradient": "roygb",
                          "min": 0.5,
                          "max": 0.9,
                      }
                  }
              })

              view.zoomTo()
              view.show()

        if os.path.exists(frame_path):
          os.remove(frame_path)

      if fail:
        break

    while is_process_running(pid):
      time.sleep(0.1)

  except KeyboardInterrupt:
    os.kill(pid, signal.SIGTERM)
    progress.bar_style = "danger"
    progress.description = "stopped"

def run_diffusion(
    contigs,
    path,
    pdb=None,
    iterations=50,
    symmetry="none",
    order=1,
    hotspot=None,
    chains=None,
    add_potential=False,
    num_designs=1,
    visual="none"):

  full_path = f"outputs/{path}"
  os.makedirs(full_path, exist_ok=True)

  opts = [
      f"inference.output_prefix={full_path}",
      f"inference.num_designs={num_designs}",
  ]

  if chains == "":
    chains = None

  # Determine symmetry.
  if symmetry in ["auto", "cyclic", "dihedral"]:
    if symmetry == "auto":
      sym = None
      copies = 1
    else:
      sym, copies = {
          "cyclic": (
              f"c{order}",
              order),
          "dihedral": (
              f"d{order}",
              order * 2),
      }[symmetry]

  else:
    symmetry = None
    sym = None
    copies = 1

  # Determine diffusion mode.
  contigs = (
      contigs
      .replace(",", " ")
      .replace(":", " ")
      .split()
  )

  is_fixed = False
  is_free = False
  fixed_chains = []

  for contig in contigs:
    for segment in contig.split("/"):
      prefix = segment.split("-")[0]

      if prefix[0].isalpha():
        is_fixed = True

        if prefix[0] not in fixed_chains:
          fixed_chains.append(prefix[0])

      if prefix.isnumeric():
        is_free = True

  if len(contigs) == 0 or not is_free:
    mode = "partial"
  elif is_fixed:
    mode = "fixed"
  else:
    mode = "free"

  # Process fixed input structures.
  if mode in ["partial", "fixed"]:
    pdb_str = pdb_to_string(
        get_pdb(pdb),
        chains=chains)

    if symmetry == "auto":
      result, pdb_str = run_ananas(
          pdb_str,
          path)

      if result is None:
        print("ERROR: no symmetry detected")
        symmetry = None
        sym = None
        copies = 1

      elif result["group"][0] == "c":
        symmetry = "cyclic"
        sym = result["group"]
        copies = int(result["group"][1:])

      elif result["group"][0] == "d":
        symmetry = "dihedral"
        sym = result["group"]
        copies = 2 * int(result["group"][1:])

      else:
        print(
            "ERROR: the detected symmetry "
            f"({result['group']}) is not supported")

        symmetry = None
        sym = None
        copies = 1

    elif mode == "fixed":
      pdb_str = pdb_to_string(
          pdb_str,
          chains=fixed_chains)

    pdb_filename = (
        f"{full_path}/input.pdb")

    with open(pdb_filename, "w") as handle:
      handle.write(pdb_str)

    parsed_pdb = parse_pdb(pdb_filename)

    opts.append(
        f"inference.input_pdb={pdb_filename}")

    if mode == "partial":
      iterations = int(
          80 * (iterations / 200))

      opts.append(
          f"diffuser.partial_T={iterations}")

      contigs = fix_partial_contigs(
          contigs,
          parsed_pdb)

    else:
      opts.append(
          f"diffuser.T={iterations}")

      contigs = fix_contigs(
          contigs,
          parsed_pdb)

  else:
    opts.append(
        f"diffuser.T={iterations}")

    parsed_pdb = None

    contigs = fix_contigs(
        contigs,
        parsed_pdb)

  if hotspot is not None and hotspot != "":
    opts.append(
        f"ppi.hotspot_res=[{hotspot}]")

  # Symmetry options.
  if sym is not None:
    sym_opts = [
        "--config-name symmetry",
        f"inference.symmetry={sym}",
    ]

    if add_potential:
      sym_opts += [
          "'potentials.guiding_potentials="
          "[\"type:olig_contacts,"
          "weight_intra:1,"
          "weight_inter:0.1\"]'",

          "potentials.olig_intra_all=True",
          "potentials.olig_inter_all=True",
          "potentials.guide_scale=2",
          "potentials.guide_decay=quadratic",
      ]

    opts = sym_opts + opts
    contigs = sum([contigs] * copies, [])

  opts.append(
      f"'contigmap.contigs="
      f"[{' '.join(contigs)}]'")

  opts += [
      "inference.dump_pdb=True",
      "inference.dump_pdb_path='/dev/shm'",
  ]

  print("mode:", mode)
  print("output:", full_path)
  print("contigs:", contigs)

  opts_str = " ".join(opts)

  command = (
      f"{sys.executable} "
      f"RFdiffusion/run_inference.py "
      f"{opts_str}"
  )

  print(command)

  run(
      command,
      iterations,
      num_designs,
      visual=visual)

  # Repair output PDB numbering and contigs.
  for design_number in range(num_designs):
    pdb_files = [
        f"outputs/traj/"
        f"{path}_{design_number}_pX0_traj.pdb",

        f"outputs/traj/"
        f"{path}_{design_number}_Xt-1_traj.pdb",

        f"{full_path}_{design_number}.pdb",
    ]

    for pdb_file in pdb_files:
      with open(pdb_file, "r") as handle:
        pdb_str = handle.read()

      with open(pdb_file, "w") as handle:
        handle.write(
            fix_pdb(
                pdb_str,
                contigs))

  return contigs, copies

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

$ /usr/bin/python3 -m pip install -q --upgrade --force-reinstall torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124

$ /usr/bin/python3 -m pip install -q --upgrade --force-reinstall --no-dependencies dgl==2.4.0+cu124 -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

$ /usr/bin/python3 -m pip install -q numpy==2.0.2 scipy==1.16.3 jedi omegaconf hydra-core icecream pyrsistent pynvml decorator e3nn==0.5.5 opt_einsum_fx py3Dmol absl-py biopython chex dm-haiku dm-tree immutabledict ml-collections optax joblib

$ /usr/bin/python3 -m pip install -q --upgrade git+https://github.com/NVIDIA/dllogger#egg=dllogger

$ git clone https://github.com/sokrypton/RFdiffusion.git RFdiffusion

$ git fetch --depth 1 origin 597d37f2a686e23941440fddf6daa4cb778e7bc7

$ git checkout --detach 597d37f2a686e23941440fddf6daa4cb778e7bc7

$ /usr/bin/python3 -m pip install -q --upgrade --force-reinstall --no-de

<timed exec>:320: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.



Validated environment
Python: 3.12.13
PyTorch: 2.4.0+cu124
DGL: 2.4.0+cu124
GPU: Tesla T4
Base_ckpt.pt: 483,616,107 bytes sha256=0fcf7d7c32b4848030aca3a051e6768de194616f96ba6c38186351a33bfc6eca
Complex_base_ckpt.pt: 483,619,179 bytes sha256=76e4e260aefee3b582bd76b77ab95d2592e64f00c51bf344968ab9239f3250bc
Complex_beta_ckpt.pt: 483,380,617 bytes sha256=5a0b1cafc23c60b1aabcec1e49391986ac4fd02cc1b6b4cc41714ca9fe882e9e
RFdiffusion 50-step GPU smoke test: PASS
CPU times: user 11.7 s, sys: 8.23 s, total: 19.9 s
Wall time: 7min 7s


In [ ]:
%%time
#@title run **RFdiffusion** to generate a backbone
name = "test" #@param {type:"string"}
contigs = "A8-175:70-100" #@param {type:"string"}
pdb = "1YZK" #@param {type:"string"}
iterations = 50 #@param ["25", "50", "100", "150", "200"] {type:"raw"}
hotspot = "A38,A41,A91,A128" #@param {type:"string"}
num_designs = 4 #@param ["1", "2", "4", "8", "16", "32"] {type:"raw"}
visual = "image" #@param ["none", "image", "interactive"]
#@markdown ---
#@markdown **symmetry** settings
#@markdown ---
symmetry = "none" #@param ["none", "cyclic", "dihedral"]
order = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"] {type:"raw"}
chains = "A" #@param {type:"string"}
add_potential = True #@param {type:"boolean"}
#@markdown - `symmetry='auto'` enables automatic symmetry dectection with [AnAnaS](https://team.inria.fr/nano-d/software/ananas/).
#@markdown - `chains="A,B"` filter PDB input to these chains (may help auto-symm detector)
#@markdown - `add_potential` to discourage clashes between chains

# determine where to save
path = name
while os.path.exists(f"outputs/{path}_0.pdb"):
  path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

flags = {"contigs":contigs,
         "pdb":pdb,
         "order":order,
         "iterations":iterations,
         "symmetry":symmetry,
         "hotspot":hotspot,
         "path":path,
         "chains":chains,
         "add_potential":add_potential,
         "num_designs":num_designs,
         "visual":visual}

for k,v in flags.items():
  if isinstance(v,str):
    flags[k] = v.replace("'","").replace('"','')

contigs, copies = run_diffusion(**flags)


$ aria2c --console-log-level=warn --summary-interval=0 --allow-overwrite=true --auto-file-renaming=false --check-integrity=true -x 16 -s 16 -d . -o 1YZK.pdb1.gz https://files.rcsb.org/download/1YZK.pdb1.gz
mode: fixed
output: outputs/test
contigs: ['A8-175', '94-94']
/usr/bin/python3 RFdiffusion/run_inference.py inference.output_prefix=outputs/test inference.num_designs=4 inference.input_pdb=outputs/test/input.pdb diffuser.T=50 ppi.hotspot_res=[A38,A41,A91,A128] 'contigmap.contigs=[A8-175 94-94]' inference.dump_pdb=True inference.dump_pdb_path='/dev/shm'


In [ ]:
#@title Display 3D structure {run: "auto"}
animate = "none" #@param ["none", "movie", "interactive"]
color = "chain" #@param ["rainbow", "chain", "plddt"]
denoise = True
dpi = 100 #@param ["100", "200", "400"] {type:"raw"}
from colabdesign.shared.plot import pymol_color_list
from colabdesign.rf.utils import get_ca, get_Ls, make_animation
from string import ascii_uppercase,ascii_lowercase
alphabet_list = list(ascii_uppercase+ascii_lowercase)

def plot_pdb(num=0):
  if denoise:
    pdb_traj = f"outputs/traj/{path}_{num}_pX0_traj.pdb"
  else:
    pdb_traj = f"outputs/traj/{path}_{num}_Xt-1_traj.pdb"
  if animate in ["none","interactive"]:
    hbondCutoff = 4.0
    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    if animate == "interactive":
      pdb_str = open(pdb_traj,'r').read()
      view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    else:
      pdb = f"outputs/{path}_{num}.pdb"
      pdb_str = open(pdb,'r').read()
      view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    if color == "rainbow":
      view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
      for n,chain,c in zip(range(len(contigs)),
                              alphabet_list,
                              pymol_color_list):
          view.setStyle({'chain':chain},{'cartoon': {'color':c}})
    else:
      view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
    view.zoomTo()
    if animate == "interactive":
      view.animate({'loop': 'backAndForth'})
    view.show()
  else:
    Ls = get_Ls(contigs)
    xyz, bfact = get_ca(pdb_traj, get_bfact=True)
    xyz = xyz.reshape((-1,sum(Ls),3))[::-1]
    bfact = bfact.reshape((-1,sum(Ls)))[::-1]
    if color == "chain":
      display(HTML(make_animation(xyz, Ls=Ls, dpi=dpi, ref=-1)))
    elif color == "rainbow":
      display(HTML(make_animation(xyz, dpi=dpi, ref=-1)))
    else:
      display(HTML(make_animation(xyz, plddt=bfact*100, dpi=dpi, ref=-1)))


if num_designs > 1:
  output = widgets.Output()
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
      options=[(f'{k}',k) for k in range(num_designs)],
      value=0, description='design:',
  )
  dropdown.observe(on_change)
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

In [ ]:
%%time
#@title run **ProteinMPNN** to generate a sequence and **AlphaFold** to validate

import os
import pathlib
import collections
import collections.abc
import pandas as pd
from IPython.display import display

# Python 3.12 compatibility used by this notebook/setup.
for _name in ("Mapping", "Iterable", "MutableMapping", "Sequence"):
    if not hasattr(collections, _name):
        setattr(collections, _name, getattr(collections.abc, _name))

num_seqs = 4 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
initial_guess = True #@param {type:"boolean"}
num_recycles = 3 #@param ["0", "1", "2", "3", "6", "12"] {type:"raw"}
use_multimer = True #@param {type:"boolean"}
rm_aa = "CW" #@param {type:"string"}
mpnn_sampling_temp = 0.1 #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5", "1.0"] {type:"raw"}

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

if not os.path.isfile("params/done.txt"):
    raise RuntimeError(
        "AlphaFold parameters are missing. "
        "The setup cell did not finish correctly."
    )

missing = [
    f"outputs/{path}_{m}.pdb"
    for m in range(num_designs)
    if not os.path.isfile(f"outputs/{path}_{m}.pdb")
]

if missing:
    raise FileNotFoundError(
        "Missing RFdiffusion structures:\n" + "\n".join(missing)
    )

result_dir = pathlib.Path("outputs") / path
result_dir.mkdir(parents=True, exist_ok=True)

contigs_str = ":".join(contigs)

args = [
    f"--pdb=outputs/{path}_0.pdb",
    f"--loc=outputs/{path}",
    f"--contigs={contigs_str}",
    f"--copies={copies}",
    f"--num_seqs={num_seqs}",
    f"--num_recycles={num_recycles}",
    f"--rm_aa={rm_aa}",
    f"--mpnn_sampling_temp={mpnn_sampling_temp}",
    f"--num_designs={num_designs}",
]

if initial_guess:
    args.append("--initial_guess")

if use_multimer:
    args.append("--use_multimer")

print("Running ProteinMPNN + AlphaFold validation...")
print("PDB:", f"outputs/{path}_0.pdb")
print("Contigs:", contigs_str)
print()

# ------------------------------------------------------------
# IMPORTANT:
# Run ColabDesign inside the notebook Python process.
# Do NOT spawn /usr/bin/python3.
# ------------------------------------------------------------

from colabdesign.rf.designability_test import main as designability_main

designability_main(args)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

results_csv = result_dir / "mpnn_results.csv"
best_pdb = result_dir / "best.pdb"

if not results_csv.is_file():
    raise RuntimeError(
        f"ProteinMPNN/AlphaFold finished but {results_csv} was not created."
    )

if not best_pdb.is_file():
    raise RuntimeError(
        f"ProteinMPNN/AlphaFold finished but {best_pdb} was not created."
    )

df = pd.read_csv(results_csv, index_col=0)

# Best first, following the script's RMSD criterion.
if "rmsd" in df.columns:
    df = df.sort_values("rmsd").reset_index(drop=True)

df.insert(0, "rank", range(1, len(df) + 1))

show = df.copy()

for col in ["mpnn", "plddt", "ptm", "i_ptm", "pae", "i_pae", "rmsd"]:
    if col in show.columns:
        show[col] = pd.to_numeric(show[col], errors="coerce").round(3)

print()
print("VALIDATION COMPLETE")
print("Results:", results_csv)
print("Best PDB:", best_pdb)
print()

display(show)

In [ ]:
#@title Display best result
import py3Dmol
def plot_pdb(num = "best"):
  if num == "best":
    with open(f"outputs/{path}/best.pdb","r") as f:
      # REMARK 001 design {m} N {n} RMSD {rmsd}
      info = f.readline().strip('\n').split()
    num = info[3]
  hbondCutoff = 4.0
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
  pdb_str = open(f"outputs/{path}_{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  pdb_str = open(f"outputs/{path}/best_design{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})

  view.setStyle({"model":0},{'cartoon':{}}) #: {'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.setStyle({"model":1},{'cartoon':{'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.zoomTo()
  view.show()

if num_designs > 1:
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
    options=["best"] + [str(k) for k in range(num_designs)],
    value="best",
    description='design:',
  )
  dropdown.observe(on_change)
  output = widgets.Output()
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

In [ ]:
#@title Package and download results
#@markdown If you are having issues downloading the result archive,
#@markdown try disabling your adblocker and run this cell again.
#@markdown  If that fails click on the little folder icon to the
#@markdown  left, navigate to file: `name.result.zip`,
#@markdown  right-click and select \"Download\"
#@markdown (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).
!zip -r {path}.result.zip outputs/{path}* outputs/traj/{path}*
files.download(f"{path}.result.zip")

**Instructions**
---
---

Use `contigs` to define continious chains. Use a `:` to define multiple contigs and a `/` to define mutliple segments within a contig.
For example:

**unconditional**
- `contigs='100'` - diffuse **monomer** of length 100
- `contigs='50:100'` - diffuse **hetero-oligomer** of lengths 50 and 100
- `contigs='50'` `symmetry='cyclic'` `order=2` - make two copies of the defined contig(s) and add a symmetry constraint, for **homo-oligomeric** diffusion.

**binder design**
- `contigs='A:50'` `pdb='4N5T'` - diffuse a **binder** of length 50 to chain A of defined PDB.
- `contigs='E6-155:70-100'` `pdb='5KQV'` `hotspot='E64,E88,E96'` - diffuse a **binder** of length 70 to 100 (sampled randomly) to chain E and defined hotspot(s).

**motif scaffolding**
 - `contigs='40/A163-181/40'` `pdb='5TPN'`
 - `contigs='A3-30/36/A33-68'` `pdb='6MRR'` - diffuse a loop of length 36 between two segments of defined PDB ranges.

**partial diffusion**
- `contigs=''` `pdb='6MRR'` - noise all coordinates
- `contigs='A1-10'` `pdb='6MRR'` - keep first 10 positions fixed, noise the rest
- `contigs='A'` `pdb='1SSC'` - fix chain A, noise the rest

*hints and tips*
- `pdb=''` leave blank to get an upload prompt
- `contigs='50-100'` use dash to specify a range of lengths to sample from